# Sepsis (Ye 2026, GSE279452): cell-type UMAPs of the three integrated datasets
*2026-10-16*

| Panel | Dataset | Integration | Cells shown | Labels |
|---|---|---|---|---|
| Real | GSE279452 recap, 133 samples | Harmony, batch = **sample** (10 corrected PCs) | all in-paper-scheme cells | Ye 2026 published scheme (major / fine) |
| K = 1,000 | PBISC bulk-only LoRA-5, 122 samples × 5 draws | Harmony, batch = **draw** | Fig 1: 61,000 (100 / sample / draw); Fig 2: 305,000 (500 / sample / draw) | Fig 1: Ye 2026 major via kNN transfer from real cells; Fig 2: model-native AIFI L2 |
| K = 4,000 | same, K = 4,000 recipe | Harmony, batch = **draw** | same | same |

**Sources (nothing recomputed here; this notebook only reads stored embeddings and labels):**
- Real: `original_scRNA/sepsis_GSE279452_full/sepsis_GSE279452_recap.h5ad` → `obsm['X_umap']`, `obs['celltype_major_ye2026' | 'celltype_ye2026' | 'in_paper_scheme']` (extracted to `data/sepsis_real_umap_obs.npz` by `extract_umap_obs.py`).
- Generated, Fig 1: `integration_report_20261012/results/generated_umaps.npz` (`sepsis_K_umap_post`) + `results/sepsis_K_transfer.csv.gz` (kNN k = 30 transfer in the real/generated joint Harmony space, `code/c_joint.py`). Rows of the two files are aligned.
- Generated, Fig 2: `harmony/sepsis_k1000/sepsis_k1000_harmony_clustering.h5ad` and the K = 4,000 counterpart → `obsm['X_umap']`, `obs['aifi_l2' | 'celltype_l1_pred']` (extracted to `data/sepsis_K_umap_obs.npz`).

Note: the Fig 1 generated UMAPs are the integration-report embeddings (Harmony on the 100-cells-per-sample-per-draw subset); the Fig 2 UMAPs are the full 305k-cell embeddings. They are separate UMAP runs, so coordinates differ between the two figures.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.colors as mc, matplotlib.patheffects as pe
import anndata as ad, logging, warnings
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR); warnings.filterwarnings("ignore", module="anndata")
from IPython.display import Image, display
BASE = os.environ.get("PBISC_BASE", "F:/PBISC")
REP = f"{BASE}/bulkonly_lora5_20261006/analysis/integration_report_20261012"
OUT = f"{REP}/celltype_umap_20261016"; DATA = f"{OUT}/data"; FIG = f"{OUT}/figs"; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"font.family": ["Arial", "Liberation Sans"], "font.size": 9, "axes.titlesize": 10,
                     "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.dpi": 300, "pdf.fonttype": 42})
def cat(z, c): return z[c + "__cats"][z[c + "__codes"]].astype(str)

## Load

In [ ]:
zr = np.load(f"{DATA}/sepsis_real_umap_obs.npz", allow_pickle=True)
keep = zr["in_paper_scheme__raw"].astype(bool)
real = pd.DataFrame({"major": cat(zr, "celltype_major_ye2026"), "fine": cat(zr, "celltype_ye2026"),
                     "sample": cat(zr, "sample"), "group": cat(zr, "group"), "age_group": cat(zr, "age_group")})[keep].reset_index(drop=True)
real_umap = zr["umap"][keep]
print(f"real: {len(keep):,} QC-passed cells, {keep.sum():,} in paper scheme, {real['sample'].nunique()} samples")

G = np.load(f"{REP}/results/generated_umaps.npz")
sub, full = {}, {}
for K in ["k1000", "k4000"]:
    t = pd.read_csv(f"{REP}/results/sepsis_{K}_transfer.csv.gz", keep_default_na=False)
    u = G[f"sepsis_{K}_umap_post"]; assert len(t) == len(u)
    sub[K] = (u, t)
    z = np.load(f"{DATA}/sepsis_{K}_umap_obs.npz", allow_pickle=True)
    full[K] = (z["umap"], pd.DataFrame({"aifi_l2": cat(z, "aifi_l2"), "l1_pred": cat(z, "celltype_l1_pred"),
                                        "draw": cat(z, "draw"), "sample_id": cat(z, "sample_id"), "group": cat(z, "group")}))
    print(f"{K}: subset {len(t):,} cells ({t.sample_id.nunique()} samples × {t.draw.nunique()} draws); full {len(z['umap']):,} cells")

## Colours
Eight categorical slots, assigned by lineage so related types sit in one hue family; every cluster is also labelled directly on the map, so identity never rests on colour alone (with ≥ 4 categories in a scatter no palette separates every pair).

In [ ]:
MAJ = {"CD4+ T cells": "#2a78d6", "CD8+ T cells": "#1baf7a", "NKT cells": "#6250d6", "NK cells": "#008300",
       "B cells": "#eda100", "Plasma cells": "#e87ba4", "Monocytes": "#e34948", "Macrophages": "#8a5a2b",
       "DC": "#eb6834", "Other cells": "#9a9a9a"}
MAJ_ORDER = list(MAJ)
# AIFI L2 (model-native) -> the same lineage families
AIFI_FAM = {"Naive CD4 T cell": "CD4+ T cells", "Memory CD4 T cell": "CD4+ T cells", "Treg": "CD4+ T cells",
            "Naive CD8 T cell": "CD8+ T cells", "Memory CD8 T cell": "CD8+ T cells", "CD8aa": "CD8+ T cells",
            "MAIT": "NKT cells", "gdT": "NKT cells", "DN T cell": "NKT cells", "Proliferating T cell": "CD8+ T cells",
            "CD56dim NK cell": "NK cells", "CD56bright NK cell": "NK cells", "Proliferating NK cell": "NK cells", "ILC": "NK cells",
            "Naive B cell": "B cells", "Memory B cell": "B cells", "Transitional B cell": "B cells", "Effector B cell": "B cells",
            "Plasma cell": "Plasma cells", "CD14 monocyte": "Monocytes", "CD16 monocyte": "Monocytes", "Intermediate monocyte": "Monocytes",
            "cDC1": "DC", "cDC2": "DC", "pDC": "DC", "ASDC": "DC", "Platelet": "Other cells", "Erythrocyte": "Other cells", "Progenitor cell": "Other cells"}
def shades(base, n):
    """n tints/shades of one hue, darkest to lightest, all kept away from white."""
    if n == 1: return [base]
    rgb = np.array(mc.to_rgb(base)); out = []
    for f in np.linspace(-0.45, 0.45, n):
        out.append(mc.to_hex(rgb * (1 + f) if f < 0 else rgb + (1 - rgb) * f))
    return out
def family_colors(labels, fam_of):
    labs = sorted(pd.unique(labels), key=lambda l: (MAJ_ORDER.index(fam_of[l]), l)); col = {}
    for f in MAJ_ORDER:
        members = [l for l in labs if fam_of[l] == f]
        col.update(dict(zip(members, shades(MAJ[f], len(members)))))
    return labs, col

## Plot helpers

In [ ]:
HALO = [pe.withStroke(linewidth=2.2, foreground="white")]
def scatter(ax, U, lab, colors, s, title, min_label=40, text=None, fs=7):
    lab = np.asarray(lab); idx = np.random.default_rng(0).permutation(len(U))
    ax.scatter(U[idx, 0], U[idx, 1], c=[colors.get(x, "#cccccc") for x in lab[idx]], s=s, linewidths=0, rasterized=True)
    for c in colors:
        m = lab == c
        if m.sum() >= min_label:
            xy = np.median(U[m], 0); ax.text(*xy, text(c) if text else c, fontsize=fs, ha="center", va="center", color="#1f1f1f", path_effects=HALO)
    ax.set_title(title, loc="left"); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal", adjustable="datalim")
    for sp in ax.spines.values(): sp.set_visible(False)
def umap_axes(ax):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim(); L = 0.12
    ax.annotate("", xy=(x0 + L * (x1 - x0), y0), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", lw=0.8, color="#555"))
    ax.annotate("", xy=(x0, y0 + L * (y1 - y0)), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", lw=0.8, color="#555"))
    ax.text(x0, y0 - 0.03 * (y1 - y0), "UMAP1", fontsize=7, color="#555", va="top"); ax.text(x0 - 0.03 * (x1 - x0), y0, "UMAP2", fontsize=7, color="#555", rotation=90, ha="right")
SHORT = {"CD4+ T cells": "CD4 T", "CD8+ T cells": "CD8 T", "NKT cells": "NKT", "NK cells": "NK", "B cells": "B", "Plasma cells": "Plasma",
         "Monocytes": "Mono", "Macrophages": "Macro", "DC": "DC", "Other cells": "Other"}

## Figure 1 — major cell types, one label scheme across all three datasets

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5.9))
panels = [("Real — GSE279452", f"133 samples · Harmony by sample · {len(real):,} cells", real_umap, real["major"].to_numpy(), 0.05)]
for K, name in [("k1000", "PBISC K = 1,000"), ("k4000", "PBISC K = 4,000")]:
    u, t = sub[K]
    panels.append((name, f"122 samples × 5 draws · Harmony by draw · {len(t):,} cells", u, t["major"].to_numpy(), 0.5))
comp = {}
for ax, (name, sub_t, U, lab, s) in zip(axs, panels):
    scatter(ax, U, lab, MAJ, s, f"{name}\n{sub_t}", min_label=60, text=lambda c: SHORT[c], fs=8); umap_axes(ax)
    comp[name] = pd.Series(lab).value_counts(normalize=True).reindex(MAJ_ORDER).fillna(0) * 100
h = [plt.Line2D([], [], marker="o", ls="", ms=6, color=MAJ[c], label=c) for c in MAJ_ORDER]
fig.legend(handles=h, loc="lower center", ncol=10, frameon=False, fontsize=8.5, handletextpad=0.2, columnspacing=1.2, bbox_to_anchor=(0.5, -0.01))
fig.text(0.01, 0.995, "Sepsis (Ye 2026): cell-type UMAPs of the real and PBISC-generated integrations", fontsize=12, weight="bold", va="top")
fig.text(0.01, 0.955, "Real cells: published major categories. Generated cells: the same categories transferred by kNN (k = 30) from real cells in a joint Harmony space; each generated panel is its own draw-integrated UMAP (100 cells / sample / draw).",
         fontsize=8, color="#555", va="top")
plt.tight_layout(rect=(0, 0.04, 1, 0.93)); fig.savefig(f"{FIG}/sepsis_celltype_umap_major.png"); fig.savefig(f"{FIG}/sepsis_celltype_umap_major.pdf"); plt.close(fig)
display(Image(f"{FIG}/sepsis_celltype_umap_major.png", width=1100))
comp = pd.DataFrame(comp).round(1); comp.to_csv(f"{FIG}/sepsis_major_composition_pct.csv"); comp

## Figure 2 — fine labels, each dataset in its native scheme
Real: Ye 2026 subsets. Generated: PBISC's own per-cell AIFI L2 call on the full 305,000-cell draw-integrated embedding. Numbers on the map refer to the legend under each panel; colours follow the lineage families of Figure 1.

In [ ]:
real_fam = real.groupby("fine")["major"].agg(lambda s: s.value_counts().index[0]).to_dict()
real_fam = {k: (v if v in MAJ else "Other cells") for k, v in real_fam.items()}
pan = [("Real — GSE279452", f"Ye 2026 fine subsets · {len(real):,} cells", real_umap, real["fine"].to_numpy(), real_fam, 0.05)]
for K, name in [("k1000", "PBISC K = 1,000"), ("k4000", "PBISC K = 4,000")]:
    u, o = full[K]; pan.append((name, f"AIFI L2 (model-native) · {len(o):,} cells", u, o["aifi_l2"].to_numpy(), AIFI_FAM, 0.12))
fig = plt.figure(figsize=(18, 9.6)); gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.62], hspace=0.02, wspace=0.06)
for j, (name, sub_t, U, lab, fam, s) in enumerate(pan):
    labs, col = family_colors(lab, fam); num = {l: str(i + 1) for i, l in enumerate(labs)}
    ax = fig.add_subplot(gs[0, j]); scatter(ax, U, lab, col, s, f"{name}\n{sub_t}", min_label=80, text=lambda c: num[c], fs=7.5)
    lax = fig.add_subplot(gs[1, j]); lax.axis("off"); n = pd.Series(lab).value_counts()
    items = [f"{num[l]:>2}  {l}  ({n.get(l, 0) / len(lab) * 100:.1f}%)" for l in labs]
    ncol = 2 if len(items) > 26 else 1; per = int(np.ceil(len(items) / ncol))
    for c in range(ncol):
        for r, (txt, l) in enumerate(zip(items[c * per:(c + 1) * per], labs[c * per:(c + 1) * per])):
            y = 1 - r * (1 / max(per, 26)); x = 0.02 + c * 0.5
            lax.scatter([x], [y], s=22, color=col[l], transform=lax.transAxes, clip_on=False)
            lax.text(x + 0.025, y, txt, fontsize=6.6 if ncol == 2 else 7.2, va="center", transform=lax.transAxes)
fig.suptitle("Sepsis (Ye 2026): fine cell-type labels on each integrated dataset", x=0.125, ha="left", y=0.985, fontsize=12, weight="bold")
fig.savefig(f"{FIG}/sepsis_celltype_umap_fine.png", bbox_inches="tight"); fig.savefig(f"{FIG}/sepsis_celltype_umap_fine.pdf", bbox_inches="tight"); plt.close(fig)
display(Image(f"{FIG}/sepsis_celltype_umap_fine.png", width=1100))

## Lightweight .h5ad copies (embeddings + labels only, no expression)
For reopening in scanpy without the 0.8–11 GB source files.

In [ ]:
def save(U, obs, path):
    obs = obs.reset_index(drop=True).astype(str).astype("category"); obs.index = obs.index.astype(str); a = ad.AnnData(obs=obs); a.obsm["X_umap"] = np.asarray(U, np.float32); a.write_h5ad(path, compression="gzip")
save(real_umap, real, f"{DATA}/sepsis_real_umap_labels.h5ad")
for K in ["k1000", "k4000"]:
    u, t = sub[K]; save(u, t[["sample_id", "draw", "group", "celltype_l1_pred", "aifi_l2", "label", "major"]], f"{DATA}/sepsis_{K}_subset_umap_transfer.h5ad")
    u, o = full[K]; save(u, o, f"{DATA}/sepsis_{K}_full_umap_aifi.h5ad")
sorted(os.listdir(DATA))

## Checks

In [ ]:
# 1) transferred labels vs PBISC's own L1 call on the Fig 1 subset (are the two label sources consistent?)
L1 = {"CD4+ T cells": "T cell", "CD8+ T cells": "T cell", "NKT cells": "T cell", "NK cells": "NK cell", "B cells": "B cell", "Plasma cells": "B cell",
      "Monocytes": "Monocyte", "Macrophages": "Monocyte", "DC": "DC", "Other cells": "Other"}
for K in ["k1000", "k4000"]:
    t = sub[K][1]; print(K, "transferred-major → L1 agrees with PBISC L1 call:", f"{(t.major.map(L1) == t.celltype_l1_pred).mean():.1%}")
# 2) cell totals per panel
print({"real": len(real), **{f"{K}_subset": len(sub[K][1]) for K in sub}, **{f"{K}_full": len(full[K][1]) for K in full}})